In [1]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 160)
pd.set_option("display.max_rows", 200)   # so the full null table isn't truncated

DATA_PATH = "AmesHousing.csv"
PLOT_DIR = "eda_plots"
os.makedirs(PLOT_DIR, exist_ok=True)

sns.set_style("whitegrid")


def savefig(name):
    path = os.path.join(PLOT_DIR, name)
    plt.tight_layout()
    plt.savefig(path, dpi=150)
    print(f"  -> saved {path}")
    plt.close()

###  TASK 1 — Load, inspect

In [ ]:
df = pd.read_csv(DATA_PATH)

# Ames CSV ships with space-separated column names (e.g. "Gr Liv Area").
# Normalize once so every downstream script (Part 2/3/4) can rely on the
# same convention.
df.columns = df.columns.str.strip().str.replace(" ", "_").str.replace("/", "_")

print("First 5 rows:")
print(df.head())
print("\nColumn dtypes:")
print(df.dtypes)
print(f"\nShape: {df.shape[0]} rows x {df.shape[1]} columns")

# Keep an untouched copy for the "before imputation" comparisons required
# later in Task 8a.
df_raw = df.copy()

# Order / PID are row identifiers, not predictive features — drop them now
# and note it as a cleaning decision in the README.
id_cols = [c for c in ["Order", "PID"] if c in df.columns]
df = df.drop(columns=id_cols)
print(f"\nDropped identifier columns: {id_cols}")


### TASK 2 — Null value analysis


In [ ]:

null_counts = df.isnull().sum()
null_pct = (df.isnull().sum() / df.shape[0]) * 100
null_table_full = pd.DataFrame({
    "null_count": null_counts,
    "null_pct": null_pct
}).sort_values("null_pct", ascending=False)

print(f"Null count/percentage for ALL {df.shape[1]} columns:")
print(null_table_full)

null_table_nonzero = null_table_full[null_table_full["null_count"] > 0]
print("\nColumns with missing values (count, %) — subset of the table above:")
print(null_table_nonzero)

high_null_cols = null_table_full[null_table_full["null_pct"] > 20].index.tolist()
low_null_cols = null_table_nonzero[null_table_nonzero["null_pct"] <= 20].index.tolist()

print(f"\nColumns exceeding 20% null rate ({len(high_null_cols)}): {high_null_cols}")
print(f"Columns below 20% null rate that need filling ({len(low_null_cols)}): {low_null_cols}")

df = df.drop(columns=high_null_cols)
print(f"\nDropped high-null (>20%) columns: {high_null_cols}")


for col in low_null_cols:
    if col not in df.columns:
        continue
    if pd.api.types.is_numeric_dtype(df[col]):
        median_val = df[col].median()
        df[col] = df[col].fillna(median_val)
        print(f"  Filled numeric '{col}' nulls with median = {median_val}")
    else:
        mode_val = df[col].mode(dropna=True)[0]
        df[col] = df[col].fillna(mode_val)
        print(f"  Filled categorical '{col}' nulls with mode = '{mode_val}'")

print(f"\nRemaining nulls after Task 2 cleanup: {df.isnull().sum().sum()}")

### # TASK 3 — Duplicate detection and removal


In [ ]:
n_dupes = df.duplicated().sum()
print(f"Duplicate rows found: {n_dupes}")

null_pct_before_dedup = (df.isnull().sum() / df.shape[0]) * 100
rows_before = df.shape[0]

df = df.drop_duplicates()

rows_after = df.shape[0]
null_pct_after_dedup = (df.isnull().sum() / df.shape[0]) * 100
print(f"Rows removed: {rows_before - rows_after}")

pct_changed = not null_pct_before_dedup.equals(null_pct_after_dedup)
print(f"Did null percentages change after de-duplication? {pct_changed}")

### TASK 4 — Data type correction


In [ ]:

mem_before = df.memory_usage(deep=True).sum()
print(f"Memory usage BEFORE dtype correction: {mem_before / 1024:.2f} KB")


if "MS_SubClass" in df.columns:
    df["MS_SubClass"] = df["MS_SubClass"].astype(str).astype("category")
    print("Corrected 'MS_SubClass' from int64 (fake-numeric) -> category")


if "Neighborhood" in df.columns:
    df["Neighborhood"] = df["Neighborhood"].astype("category")
    print("Converted 'Neighborhood' (repetitive string) -> category")

mem_after = df.memory_usage(deep=True).sum()
print(f"Memory usage AFTER dtype correction:  {mem_after / 1024:.2f} KB")
print(f"Memory saved: {(mem_before - mem_after) / 1024:.2f} KB "
      f"({100 * (mem_before - mem_after) / mem_before:.1f}% reduction)")



### TASK 5 — Descriptive statistics and skewness


In [ ]:
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
print("Numeric columns:", numeric_cols)
print("\ndf.describe():")
print(df[numeric_cols].describe())

skew_series = df[numeric_cols].skew().sort_values(key=lambda s: s.abs(), ascending=False)
print("\nSkewness per numeric column (sorted by |skew|):")
print(skew_series)

most_skewed_col = skew_series.index[0]
most_skewed_val = skew_series.iloc[0]
print(f"\nMost skewed column: '{most_skewed_col}' (skew = {most_skewed_val:.3f})")

In [ ]:
### TASK 6 — Outlier detection with IQR


In [ ]:
iqr_target_cols = ["SalePrice", "Gr_Liv_Area"]
iqr_target_cols = [c for c in iqr_target_cols if c in df.columns]

outlier_summary = {}
for col in iqr_target_cols:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    mask = (df[col] < lower) | (df[col] > upper)
    n_outliers = mask.sum()
    outlier_summary[col] = {
        "Q1": Q1, "Q3": Q3, "IQR": IQR,
        "lower_bound": lower, "upper_bound": upper,
        "n_outliers": n_outliers,
    }
    print(f"\n{col}: Q1={Q1:.2f}, Q3={Q3:.2f}, IQR={IQR:.2f}, "
          f"bounds=({lower:.2f}, {upper:.2f}), outliers={n_outliers}")

### TASK 7 — Visualizations (5 required types)


In [ ]:
if {"Yr_Sold", "Mo_Sold", "SalePrice"}.issubset(df.columns):
    df_time_sorted = df.sort_values(["Yr_Sold", "Mo_Sold"])
    plt.figure(figsize=(10, 5))
    plt.plot(df_time_sorted["SalePrice"].values, linewidth=0.8)
    plt.title("SalePrice Across Transactions (Sorted by Sale Year/Month)")
    plt.xlabel("Transaction Order (sorted by Yr_Sold, Mo_Sold)")
    plt.ylabel("SalePrice ($)")
    savefig("01_line_saleprice_over_time.png")

### 7.1 Line plot

In [ ]:
if {"Yr_Sold", "Mo_Sold", "SalePrice"}.issubset(df.columns):
    df_time_sorted = df.sort_values(["Yr_Sold", "Mo_Sold"])
    plt.figure(figsize=(10, 5))
    plt.plot(df_time_sorted["SalePrice"].values, linewidth=0.8)
    plt.title("SalePrice Across Transactions (Sorted by Sale Year/Month)")
    plt.xlabel("Transaction Order (sorted by Yr_Sold, Mo_Sold)")
    plt.ylabel("SalePrice ($)")
    savefig("01_line_saleprice_over_time.png")

### 7.2 Bar chart

In [ ]:
if {"House_Style", "SalePrice"}.issubset(df.columns):
    plt.figure(figsize=(10, 5))
    df.groupby("House_Style", observed=True)["SalePrice"].mean().sort_values().plot.bar()
    plt.title("Mean SalePrice by House Style")
    plt.xlabel("House Style")
    plt.ylabel("Mean SalePrice ($)")
    savefig("02_bar_mean_saleprice_by_house_style.png")

###  7.3 Histogram

In [ ]:
plt.figure(figsize=(10, 5))
sns.histplot(df[most_skewed_col], bins=20, kde=True)
plt.title(f"Distribution of {most_skewed_col} (skew = {most_skewed_val:.2f})")
plt.xlabel(most_skewed_col)
plt.ylabel("Frequency")
savefig("03_histogram_most_skewed.png")

### 7.4 Scatter plot

In [ ]:
if {"Gr_Liv_Area", "SalePrice"}.issubset(df.columns):
    plt.figure(figsize=(10, 5))
    sns.scatterplot(data=df, x="Gr_Liv_Area", y="SalePrice", alpha=0.5)
    plt.title("Gr_Liv_Area vs SalePrice")
    plt.xlabel("Above-Grade Living Area (sq ft)")
    plt.ylabel("SalePrice ($)")
    savefig("04_scatter_grlivarea_vs_saleprice.png")


### 7.5 Box plot

In [ ]:
if {"Kitchen_Qual", "SalePrice"}.issubset(df.columns):
    order = ["Po", "Fa", "TA", "Gd", "Ex"]
    order = [o for o in order if o in df["Kitchen_Qual"].unique()]
    plt.figure(figsize=(10, 5))
    sns.boxplot(data=df, x="Kitchen_Qual", y="SalePrice", order=order)
    plt.title("SalePrice by Kitchen Quality")
    plt.xlabel("Kitchen Quality (Po < Fa < TA < Gd < Ex)")
    plt.ylabel("SalePrice ($)")
    savefig("05_boxplot_saleprice_by_kitchenqual.png")

# 7.6 Correlation heat map

In [ ]:
pearson_corr = df[numeric_cols].corr(method="pearson")
plt.figure(figsize=(16, 13))
sns.heatmap(pearson_corr, annot=False, cmap="coolwarm", center=0)
plt.title("Pearson Correlation Heat Map (Numeric Columns)")
savefig("06_correlation_heatmap.png")

### Identify the pair with highest absolute correlation 

In [ ]:
corr_unstacked = pearson_corr.where(
    ~np.eye(len(pearson_corr), dtype=bool)
).unstack().dropna()
top_pair = corr_unstacked.abs().sort_values(ascending=False).index[0]
top_pair_val = corr_unstacked[top_pair]
print(f"\nHighest |Pearson correlation| pair: {top_pair} = {top_pair_val:.3f}")

### TASK 8a — Imputation strategy comparison (mean vs median)


In [ ]:
top2_skew_cols = skew_series.index[:2].tolist()
print(f"Top-2 highest-|skew| numeric columns: {top2_skew_cols}")

imputation_plan = {}
for col in top2_skew_cols:
    raw_col = df_raw[col] if col in df_raw.columns else df[col]
    mean_val = raw_col.mean()
    median_val = raw_col.median()
    skew_val = skew_series[col]
    chosen = "median" 

    
    imputation_plan[col] = {
        "mean": mean_val, "median": median_val,
        "skew": skew_val, "chosen_strategy": chosen,
    }
    print(f"\n{col}: mean={mean_val:.3f}, median={median_val:.3f}, "
          f"skew={skew_val:.3f} -> using {chosen} for any remaining nulls")
    if df[col].isnull().sum() > 0:
        fill_value = median_val if chosen == "median" else mean_val
        df[col] = df[col].fillna(fill_value)

print("\nConfirm no nulls remain in these columns:")
print(df[top2_skew_cols].isnull().sum())

### TASK 8b — Spearman rank correlation vs Pearson


In [ ]:
spearman_corr = df[numeric_cols].corr(method="spearman")

diff = (spearman_corr - pearson_corr).abs()
diff_unstacked = diff.where(~np.eye(len(diff), dtype=bool)).unstack().dropna()
# Each pair appears twice (symmetric matrix) — keep unique pairs only
diff_unstacked = diff_unstacked[
    diff_unstacked.index.map(lambda x: x[0] < x[1])
].sort_values(ascending=False)

top3_diff_pairs = diff_unstacked.head(3)
print("Top 3 pairs by |Spearman - Pearson|:")
for (a, b), d in top3_diff_pairs.items():
    p = pearson_corr.loc[a, b]
    s = spearman_corr.loc[a, b]
    print(f"  ({a}, {b}): Pearson={p:.3f}, Spearman={s:.3f}, |diff|={d:.3f}")

print("\nFull Pearson correlation matrix:")
print(pearson_corr)
print("\nFull Spearman correlation matrix:")
print(spearman_corr)

### TASK 8c — Grouped aggregation


In [ ]:
group_cat_col = "Neighborhood"
group_num_col = "SalePrice"

if group_cat_col in df.columns and group_num_col in df.columns:
    grouped = df.groupby(group_cat_col, observed=True)[group_num_col].agg(
        ["mean", "std", "count"]
    )
    print(grouped)

    highest_mean_group = grouped["mean"].idxmax()
    highest_std_group = grouped["std"].idxmax()
    ratio = grouped["mean"].max() / grouped["mean"].min()

    print(f"\nGroup with highest mean {group_num_col}: {highest_mean_group} "
          f"({grouped['mean'].max():.2f})")
    print(f"Group with highest std {group_num_col}: {highest_std_group} "
          f"({grouped['std'].max():.2f})")
    print(f"Ratio of highest group mean to lowest group mean: {ratio:.2f}")

###  SAVE CLEANED DATASET

In [ ]:


df.to_csv("cleaned_data.csv", index=False)
print(f"Saved cleaned_data.csv with shape {df.shape}")
print("Columns:", df.columns.tolist())

